In [3]:
import os
import shutil

src_dir = None
req_file = None

# Automatically find the src folder and requirements file
for dirname, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and 'chakramodel' in dirname.lower():
        src_dir = os.path.join(dirname, 'src')
    if 'requirements.txt' in files:
        req_file = os.path.join(dirname, 'requirements.txt')

if src_dir and req_file:
    print(f"Found code at: {src_dir}")
    
    # Copy to working directory
    if os.path.exists('./src'):
        shutil.rmtree('./src')
    shutil.copytree(src_dir, './src')
    print("Copied 'src' to working directory!")
    
    # Install requirements
    print("Installing requirements...")
    os.system(f"pip install -r {req_file}")
    print("Setup complete! You can now run your training script.")
else:
    print("Error: Could not find your uploaded files. Please run the very first cell (os.walk) to see your file paths.")

Found code at: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
Copied 'src' to working directory!
Installing requirements...
Setup complete! You can now run your training script.


In [4]:
!pip install gudhi

In [5]:
%%writefile src/chakra_transformer/train_transformer.py
import argparse
import sys
import time
from pathlib import Path
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from train_pranet import KvasirSEGDataset, DiceFocalLoss
from metrics.seg_metrics import dice, iou
from transformer_segmenter import ChakraTransformerSegmenter
from topo_loss import TopologicalLoss

def main():
    parser = argparse.ArgumentParser(description="Train ChakraTransformer (Research Track)")
    parser.add_argument('--batch-size', type=int, default=16, help='Batch size for training')
    parser.add_argument('--epochs', type=int, default=50, help='Number of epochs')
    parser.add_argument('--lr', type=float, default=1e-4, help='Learning rate (AdamW)')
    parser.add_argument('--num-workers', type=int, default=2, help='Number of dataloader workers')
    args = parser.parse_args()

    print("=== ChakraTransformer Training (High-Accuracy Research Track) ===")
    print(f"Batch Size: {args.batch_size}, Epochs: {args.epochs}, LR: {args.lr}")
    print("Initializing Vision Transformer (ViT-Large)...")

    assert torch.cuda.is_available(), "CUDA must be available to train! CPU training is disabled."
    device = torch.device("cuda")
    model = ChakraTransformerSegmenter().to(device)
    
    
    optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    criterion_dice = DiceFocalLoss()
    criterion_topo = TopologicalLoss(lam=0.1)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=args.lr * 0.01)

    print(f"Model initialized on {device}. Loading dataset...")
    
    root = Path(__file__).resolve().parent.parent.parent
    images_dir = root / "data" / "kvasir-seg" / "images"
    masks_dir  = root / "data" / "kvasir-seg" / "masks"
    
    if not images_dir.exists():
        print("[ERROR] Kvasir-SEG not found. Run: python src/download_kvasir.py")
        sys.exit(1)
        
    img_size = 384  # Upgraded resolution for ViT-Large
    
    full_dataset = KvasirSEGDataset(images_dir, masks_dir, img_size=img_size, augment=True, use_gt_bbox_crop=True)
    val_dataset  = KvasirSEGDataset(images_dir, masks_dir, img_size=img_size, augment=False, use_gt_bbox_crop=True)

    all_paths = sorted([p for p in images_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif"}])
    n_train   = int(0.7 * len(all_paths))
    n_val     = int(0.1 * len(all_paths))
    n_cal     = int(0.1 * len(all_paths)) # Explicitly separate calibration/test sets

    train_set = torch.utils.data.Subset(full_dataset, range(n_train))
    val_set   = torch.utils.data.Subset(val_dataset,  range(n_train, n_train + n_val))

    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,  num_workers=args.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=True)

    weights_dir = root / "weights"
    weights_dir.mkdir(exist_ok=True)
    best_dice = 0.0
    best_path = weights_dir / "chakra_transformer_best.pth"

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss = 0.0
        
        print(f"\n--- Epoch {epoch}/{args.epochs} ---")
        pbar = tqdm(train_loader, desc="Training")
        for imgs, masks in pbar:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss_dice = criterion_dice(logits, masks)
            
            # Topological Loss is disabled in the main training loop (future work)
            # as it causes extreme slowdowns on 384x384 feature maps.
            loss = loss_dice
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        scheduler.step()
        train_loss /= len(train_loader)

        model.eval()
        val_dice_scores = []
        val_iou_scores  = []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs  = imgs.to(device)
                logits = model(imgs)
                probs  = torch.sigmoid(logits).cpu().numpy()
                masks_np = masks.numpy()
                for i in range(len(probs)):
                    pred = (probs[i, 0] > 0.5).astype(np.uint8) * 255
                    gt   = (masks_np[i, 0] > 0.5).astype(np.uint8) * 255
                    val_dice_scores.append(dice(pred, gt))
                    val_iou_scores.append(iou(pred, gt))

        mean_dice = float(np.mean(val_dice_scores))
        mean_iou  = float(np.mean(val_iou_scores))
        lr_now    = optimizer.param_groups[0]["lr"]

        print(f"  --> Epoch [{epoch:3d}/{args.epochs}] Summary | Train Loss: {train_loss:.4f} | Val Dice: {mean_dice:.4f} | Val IoU: {mean_iou:.4f} | LR: {lr_now:.6f}", flush=True)

        if mean_dice > best_dice:
            best_dice = mean_dice
            torch.save(model.state_dict(), best_path)
            print(f"  *** New best Dice: {best_dice:.4f} -> saved to {best_path}")

    print(f"\nTraining Complete! Best Val Dice: {best_dice:.4f}. Weights saved: {best_path}")

if __name__ == '__main__':
    main()


Overwriting src/chakra_transformer/train_transformer.py


In [6]:
i
m
p
o
r
t
 
o
s


#
 
C
l
e
a
n
 
u
p
 
o
l
d
 
b
r
o
k
e
n
 
l
i
n
k
s

o
s
.
s
y
s
t
e
m
(
"
r
m
 
-
r
f
 
.
/
d
a
t
a
"
)

o
s
.
m
a
k
e
d
i
r
s
(
"
.
/
d
a
t
a
"
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)


#
 
A
u
t
o
m
a
t
i
c
a
l
l
y
 
f
i
n
d
 
t
h
e
 
K
v
a
s
i
r
-
S
E
G
 
f
o
l
d
e
r

k
v
a
s
i
r
_
d
i
r
 
=
 
N
o
n
e

f
o
r
 
d
i
r
n
a
m
e
,
 
d
i
r
s
,
 
f
i
l
e
s
 
i
n
 
o
s
.
w
a
l
k
(
'
/
k
a
g
g
l
e
/
i
n
p
u
t
'
)
:

 
 
 
 
#
 
L
o
o
k
 
f
o
r
 
t
h
e
 
d
i
r
e
c
t
o
r
y
 
t
h
a
t
 
c
o
n
t
a
i
n
s
 
'
i
m
a
g
e
s
'
 
a
n
d
 
'
m
a
s
k
s
'

 
 
 
 
i
f
 
'
i
m
a
g
e
s
'
 
i
n
 
d
i
r
s
 
a
n
d
 
'
m
a
s
k
s
'
 
i
n
 
d
i
r
s
 
a
n
d
 
'
k
v
a
s
i
r
'
 
i
n
 
d
i
r
n
a
m
e
.
l
o
w
e
r
(
)
:

 
 
 
 
 
 
 
 
k
v
a
s
i
r
_
d
i
r
 
=
 
d
i
r
n
a
m
e

 
 
 
 
 
 
 
 
b
r
e
a
k


i
f
 
k
v
a
s
i
r
_
d
i
r
:

 
 
 
 
p
r
i
n
t
(
f
"
F
o
u
n
d
 
K
v
a
s
i
r
-
S
E
G
 
d
a
t
a
s
e
t
 
a
t
:
 
{
k
v
a
s
i
r
_
d
i
r
}
"
)

 
 
 
 
#
 
C
r
e
a
t
e
 
t
h
e
 
c
o
r
r
e
c
t
 
s
y
m
l
i
n
k

 
 
 
 
o
s
.
s
y
s
t
e
m
(
f
"
l
n
 
-
s
 
{
k
v
a
s
i
r
_
d
i
r
}
 
.
/
d
a
t
a
/
k
v
a
s
i
r
-
s
e
g
"
)

 
 
 
 

 
 
 
 
#
 
R
u
n
 
t
h
e
 
t
r
a
i
n
i
n
g
 
s
c
r
i
p
t

 
 
 
 
p
r
i
n
t
(
"
S
t
a
r
t
i
n
g
 
t
r
a
i
n
i
n
g
.
.
.
"
)

 
 
 
 
o
s
.
s
y
s
t
e
m
(
"
p
y
t
h
o
n
 
s
r
c
/
c
h
a
k
r
a
_
t
r
a
n
s
f
o
r
m
e
r
/
t
r
a
i
n
_
t
r
a
n
s
f
o
r
m
e
r
.
p
y
 
-
-
e
p
o
c
h
s
 
1
0
0
 
-
-
b
a
t
c
h
-
s
i
z
e
 
8
"
)

e
l
s
e
:

 
 
 
 
p
r
i
n
t
(
"
E
R
R
O
R
:
 
C
o
u
l
d
 
n
o
t
 
f
i
n
d
 
t
h
e
 
K
v
a
s
i
r
-
S
E
G
 
d
a
t
a
s
e
t
.
 
A
r
e
 
y
o
u
 
s
u
r
e
 
y
o
u
 
c
l
i
c
k
e
d
 
'
A
d
d
 
D
a
t
a
'
 
a
n
d
 
a
t
t
a
c
h
e
d
 
i
t
 
t
o
 
t
h
e
 
n
o
t
e
b
o
o
k
?
"
)

SyntaxError: unterminated string literal (detected at line 50) (2085976885.py, line 50)